# Day 09 - 天气助手：完整项目实战经过前8天的学习，你已经掌握了机器学习的核心技能！今天，我们要把这些技能**全部组合起来**，从头到尾完成一个真实的机器学习项目：**预测明天会不会下雨**。---## 今天的目标1. 回顾机器学习的完整流程2. 从原始数据到最终模型，独立完成每一步3. 保存模型，以后可以直接用

## 1. 机器学习的完整流程一个标准的机器学习项目通常包括以下步骤：```1. 收集数据  ->  2. 清洗数据  ->  3. 特征工程4. 选择模型    ->  5. 训练模型  ->  6. 评估模型7. 优化模型    ->  8. 部署使用```前面几天我们分别学了每一步，今天把它们串起来！

In [ ]:
# 导入所有需要的库import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.preprocessing import StandardScalerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import accuracy_score, classification_report, confusion_matriximport warningswarnings.filterwarnings('ignore')plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei"]plt.rcParams["axes.unicode_minus"] = Falseprint("所有库导入成功！")

## 2. 第一步：加载和清洗数据

In [ ]:
# 读取天气数据df = pd.read_csv("data/weather_sample.csv", encoding="utf-8-sig")df["日期"] = pd.to_datetime(df["日期"])"print("数据大小:", df.shape)print("")print("每列的缺失值数量:")print(df.isnull().sum())df = df.fillna(df.mean(numeric_only=True))print("清洗后数据预览:")df.head()

## 3. 第二步：特征工程这一步很关键！我们从原始数据中提取更多有用的特征。

In [ ]:
# 日期特征df["月份"] = df["日期"].dt.monthdf["星期"] = df["日期"].dt.dayofweekdf["是否周末"] = (df["星期"] >= 5).astype(int)df["年中第几天"] = df["日期"].dt.dayofyear# 季节特征def get_season(month):    if month in [3, 4, 5]: return 0    elif month in [6, 7, 8]: return 1    elif month in [9, 10, 11]: return 2    else: return 3df["季节"] = df["月份"].apply(get_season)print("特征工程后的所有列:")print(list(df.columns))df.head()

## 4. 第三步：准备训练数据

In [ ]:
# 选择特征和标签feature_cols = ["湿度", "风速", "气压", "月份", "是否周末", "季节", "温度"]X = df[feature_cols]y = df["是否下雨"]X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42)print(f"训练集大小: {X_train.shape[0]} 条数据")print(f"测试集大小: {X_test.shape[0]} 条数据")print(f"下雨比例: {y.mean()*100:.1f}%")# 标准化scaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_test_scaled = scaler.transform(X_test)print("数据已标准化！")

## 5. 第四步：训练和比较三个模型我们同时训练三个不同的模型，看看谁最好：- **逻辑回归**：最简单的分类模型- **决策树**：像规则一样思考的模型- **随机森林**：很多棵决策树一起投票

In [ ]:
# 定义三个模型models = {    "逻辑回归": LogisticRegression(random_state=42, max_iter=1000),    "决策树": DecisionTreeClassifier(random_state=42, max_depth=5),    "随机森林": RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5)}results = {}for name, model in models.items():    model.fit(X_train_scaled, y_train)    y_pred = model.predict(X_test_scaled)    acc = accuracy_score(y_test, y_pred)    results[name] = acc    print(f"{name}: 准确率 = {acc*100:.1f}%")best_name = max(results, key=results.get)print(f"最好的模型是: {best_name} ({results[best_name]*100:.1f}%)")

In [ ]:
# 可视化对比names = list(results.keys())scores = [results[n]*100 for n in names]colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']plt.figure(figsize=(8, 5))bars = plt.bar(names, scores, color=colors, edgecolor='white', linewidth=2)for bar, score in zip(bars, scores):    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,             f"{score:.1f}%", ha="center", fontsize=14, fontweight="bold")plt.ylabel("准确率 (%)", fontsize=14)plt.title("三个模型对比", fontsize=16)plt.ylim(0, 100)plt.grid(True, alpha=0.3, axis="y")plt.tight_layout()plt.show()

## 6. 第五步：深入分析最好模型让我们看看最好模型的详细表现：

In [ ]:
# 用最好的模型做详细分析best_model = models[best_name]y_pred_best = best_model.predict(X_test_scaled)print("详细分类报告:")print(classification_report(y_test, y_pred_best, target_names=["不下雨", "下雨"]))cm = confusion_matrix(y_test, y_pred_best)plt.figure(figsize=(6, 5))plt.imshow(cm, cmap='Blues')plt.title(f"{best_name} - 混淆矩阵", fontsize=16)plt.xlabel("预测", fontsize=14)plt.ylabel("真实", fontsize=14)plt.xticks([0, 1], ["不下雨", "下雨"])plt.yticks([0, 1], ["不下雨", "下雨"])for i in range(2):    for j in range(2):        plt.text(j, i, str(cm[i][j]), ha='center', va='center',                 fontsize=20, color='white' if cm[i][j] > cm.max()/2 else 'black')plt.colorbar()plt.tight_layout()plt.show()

## 7. 第六步：特征重要性分析哪些因素对预测下雨最重要？

In [ ]:
# 查看特征重要性if hasattr(best_model, 'feature_importances_'):    importances = best_model.feature_importances_else:    importances = np.abs(best_model.coef_[0])feat_imp = pd.DataFrame({"特征": feature_cols, "重要性": importances})feat_imp = feat_imp.sort_values("重要性", ascending=True)plt.figure(figsize=(8, 5))plt.barh(feat_imp["特征"], feat_imp["重要性"], color="steelblue")plt.xlabel("重要性", fontsize=14)plt.title("特征重要性排名", fontsize=16)plt.tight_layout()plt.show()print("最重要的三个特征:")print(feat_imp.tail(3).to_string(index=False))

## 8. 第七步：保存模型训练好模型后，我们可以保存它，以后就不用重新训练了！

In [ ]:
# 保存模型和标准化器import picklewith open("weather_model.pkl", "wb") as f:    pickle.dump(best_model, f)with open("weather_scaler.pkl", "wb") as f:    pickle.dump(scaler, f)print("模型已保存到 weather_model.pkl")print("标准化器已保存到 weather_scaler.pkl")# 演示：加载模型并使用with open("weather_model.pkl", "rb") as f:    loaded_model = pickle.load(f)with open("weather_scaler.pkl", "rb") as f:    loaded_scaler = pickle.load(f)print("模型加载成功！可以拿来预测了。")

---## 恭喜！你完成了第一个完整的机器学习项目！你今天做了什么：| 步骤 | 内容 ||------|------|| 1. 数据加载 | 读取 CSV 并清洗数据 || 2. 特征工程 | 从日期中提取月份、季节等特征 || 3. 数据准备 | 划分训练/测试集，标准化 || 4. 模型训练 | 同时训练了3个不同模型 || 5. 模型评估 | 比较准确率，分析混淆矩阵 || 6. 特征分析 | 找出最重要的预测因子 || 7. 模型保存 | 用 pickle 保存模型供以后使用 |---## 挑战任务（选做）1. 试试添加更多特征（如"温差"）看能否提高准确率2. 换一个城市的数据，重新训练模型3. 尝试用 GridSearchCV 自动搜索最佳参数4. 写一个函数：输入明天的天气数据，输出是否会下雨